In [0]:
import json
from pathlib import Path
name = "data/sample/410/410_DE_hour_1787522400000.json"
candidates = [Path.cwd() / name, Path.cwd().parent / name]
path = next(p for p in candidates if p.exists())
print(path)
with open(path) as f:
    data = json.load(f)
print(list(data.keys()))
print("hours", len(data["series"]))
print("first row", data["series"][0])

In [0]:
from pyspark.sql import functions as F

rows = [(410, "load", int(ts), val) for ts, val in data["series"]]
df = spark.createDataFrame(rows, ["filter_id", "metric", "event_time_ms", "value_mwh"])
df = df.withColumn("source_file", F.lit(str(path)))
df.show(5)

df.write.mode("overwrite").saveAsTable("bronze.smard_raw")
print("rows", df.count())

In [0]:
from pyspark.sql import functions as F

silver = (
    df.withColumn(
        "event_time",
        F.from_utc_timestamp(F.to_timestamp(F.col("event_time_ms") / 1000), "Europe/Berlin"),
    )
    .withColumn("is_missing_value", F.col("value_mwh").isNull())
    .dropDuplicates(["metric", "event_time"])
)

silver.show(5)
silver.write.mode("overwrite").saveAsTable("silver.electricity_hourly")
print("rows", silver.count())

In [0]:
from pyspark.sql import functions as F

gold = (
    silver.groupBy(F.to_date("event_time").alias("day"))
    .agg(
        F.count("*").alias("hours"),
        F.sum(F.when(F.col("is_missing_value"), 1).otherwise(0)).alias("missing_hours"),
        F.avg("value_mwh").alias("avg_load_mwh"),
        F.min("value_mwh").alias("min_load_mwh"),
        F.max("value_mwh").alias("max_load_mwh"),
    )
)

gold.show()
gold.write.mode("overwrite").saveAsTable("gold.daily_load")
print("days", gold.count())